In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)

In [ ]:
# Config 
DATASET_PATH  = "creditcard.csv"
MODEL_PATH    = "fraud_rf_model.joblib"
SCALER_PATH   = "fraud_scaler.joblib"
RANDOM_STATE  = 42
TEST_SIZE     = 0.20
CV_FOLDS      = 5
N_ESTIMATORS  = 100
MAX_DEPTH     = None     

In [3]:
# Load & Explore 
print("=" * 60)
print("  CREDIT CARD FRAUD DETECTION — RANDOM FOREST PIPELINE")
print("=" * 60)
 
df = pd.read_csv(DATASET_PATH)
 
print(f"\n[1] Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"    Fraud transactions : {df['Class'].sum():,}  ({df['Class'].mean()*100:.4f}%)")
print(f"    Legit transactions : {(df['Class']==0).sum():,}")
print(f"    Missing values     : {df.isnull().sum().sum()}")
print(f"\n    Amount stats:\n{df['Amount'].describe().to_string()}")

  CREDIT CARD FRAUD DETECTION — RANDOM FOREST PIPELINE

[1] Dataset loaded: 284,807 rows × 31 columns
    Fraud transactions : 492  (0.1727%)
    Legit transactions : 284,315
    Missing values     : 0

    Amount stats:
count    284807.000000
mean         88.589171
std          94.704325
min           0.010000
25%          25.225000
50%          60.980000
75%         121.770000
max        2483.940000


In [4]:
# Pre-processing
print("\n[2] Pre-processing ...")
 
# StandardScaler on Amount & Time (V1–V28 already PCA-scaled)
scaler = StandardScaler()
df["scaled_amount"] = scaler.fit_transform(df[["Amount"]])
df["scaled_time"]   = scaler.fit_transform(df[["Time"]])
 
feature_cols = (
    ["scaled_time", "scaled_amount"] +
    [f"V{i}" for i in range(1, 29)]
)
X = df[feature_cols].values
y = df["Class"].values
 
print(f"    Feature matrix : {X.shape}")
print(f"    Class balance  : {np.bincount(y)}")


[2] Pre-processing ...
    Feature matrix : (284807, 30)
    Class balance  : [284315    492]


In [ ]:
#Stratified Train / Test Split 
print("\n[3] Stratified train/test split (80/20) ...")
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)
print(f"    Train : {X_train.shape[0]:,}  |  Test : {X_test.shape[0]:,}")
print(f"    Train fraud count : {y_train.sum()}  |  Test fraud count : {y_test.sum()}")
 


[3] Stratified train/test split (80/20) ...
    Train : 227,845  |  Test : 56,962
    Train fraud count : 394  |  Test fraud count : 98


In [6]:
#Random Forest — class_weight='balanced'
print("\n[4] Training Random Forest ...")
print(f"    n_estimators={N_ESTIMATORS}, class_weight='balanced', random_state={RANDOM_STATE}")
 
clf = RandomForestClassifier(
    n_estimators   = N_ESTIMATORS,
    max_depth      = MAX_DEPTH,
    class_weight   = "balanced",      # handles severe class imbalance
    n_jobs         = -1,
    random_state   = RANDOM_STATE,
)
clf.fit(X_train, y_train)
print("    Training complete ✓")


[4] Training Random Forest ...
    n_estimators=100, class_weight='balanced', random_state=42
    Training complete ✓


In [7]:
# Stratified Cross-Validation
print(f"\n[5] {CV_FOLDS}-fold Stratified Cross-Validation on training set ...")
 
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    clf, X_train, y_train,
    cv=cv,
    scoring=["roc_auc", "average_precision"],
    n_jobs=-1,
)
 
print(f"    AUC-ROC  : {cv_results['test_roc_auc'].mean():.4f} "
      f"± {cv_results['test_roc_auc'].std():.4f}")
print(f"    Avg Prec : {cv_results['test_average_precision'].mean():.4f} "
      f"± {cv_results['test_average_precision'].std():.4f}")


[5] 5-fold Stratified Cross-Validation on training set ...
    AUC-ROC  : 1.0000 ± 0.0000
    Avg Prec : 0.9957 ± 0.0041


In [8]:
#Evaluation on Hold-out Test Set
print("\n[6] Evaluating on test set ...")
 
y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = clf.predict(X_test)
 
auc_roc   = roc_auc_score(y_test, y_prob)
avg_prec  = average_precision_score(y_test, y_prob)
cm        = confusion_matrix(y_test, y_pred)
 
print(f"\n    AUC-ROC            : {auc_roc:.4f}")
print(f"    Average Precision  : {avg_prec:.4f}")
print(f"\n    Confusion Matrix:\n{cm}")
print(f"\n    Classification Report:\n{classification_report(y_test, y_pred, target_names=['Legit','Fraud'])}")


[6] Evaluating on test set ...

    AUC-ROC            : 1.0000
    Average Precision  : 0.9897

    Confusion Matrix:
[[56864     0]
 [   78    20]]

    Classification Report:
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00     56864
       Fraud       1.00      0.20      0.34        98

    accuracy                           1.00     56962
   macro avg       1.00      0.60      0.67     56962
weighted avg       1.00      1.00      1.00     56962



In [9]:
#Feature Importance 
importances   = clf.feature_importances_
feat_series   = pd.Series(importances, index=feature_cols).sort_values(ascending=False)
print("\n[7] Top-10 Feature Importances:")
print(feat_series.head(10).to_string())


[7] Top-10 Feature Importances:
V1     0.138674
V10    0.099476
V12    0.081501
V11    0.077190
V27    0.075374
V25    0.067674
V2     0.065900
V17    0.064382
V7     0.045210
V23    0.039337


In [10]:
#Plots
print("\n[8] Generating evaluation plots ...")
 
fig = plt.figure(figsize=(18, 12))
fig.suptitle("Credit Card Fraud Detection — Random Forest", fontsize=15, fontweight="bold")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)


[8] Generating evaluation plots ...


<Figure size 1800x1200 with 0 Axes>

In [11]:
ax1 = fig.add_subplot(gs[0, 0])
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax1, color="steelblue")
ax1.set_title(f"ROC Curve  (AUC = {auc_roc:.4f})")
ax1.plot([0,1],[0,1],"k--", lw=1)

c:\Users\Sahil Singh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\_plotting.py:176: FutureWarning: `**kwargs` is deprecated and will be removed in 1.9. Pass all matplotlib arguments to `curve_kwargs` as a dictionary instead.
  warnings.warn(


In [12]:
ax2 = fig.add_subplot(gs[0, 1])
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=ax2, color="darkorange")
ax2.set_title(f"Precision-Recall Curve  (AP = {avg_prec:.4f})")

Text(0.5, 1.0, 'Precision-Recall Curve  (AP = 0.9897)')

In [13]:
ax3 = fig.add_subplot(gs[0, 2])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit","Fraud"]).plot(ax=ax3, colorbar=False)
ax3.set_title("Confusion Matrix")

Text(0.5, 1.0, 'Confusion Matrix')

In [14]:
ax4 = fig.add_subplot(gs[1, :2])
top15 = feat_series.head(15)
bars = ax4.barh(top15.index[::-1], top15.values[::-1], color="mediumseagreen")
ax4.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
ax4.set_xlabel("Importance")
ax4.set_title("Top-15 Feature Importances")

Text(0.5, 1.0, 'Top-15 Feature Importances')

In [15]:
ax5 = fig.add_subplot(gs[1, 2])
scores_df = pd.DataFrame({
    "AUC-ROC" : cv_results["test_roc_auc"],
    "Avg Prec": cv_results["test_average_precision"],
})
ax5.boxplot(scores_df.values, labels=scores_df.columns, patch_artist=True,
            boxprops=dict(facecolor="lightskyblue"), medianprops=dict(color="red", lw=2))
ax5.set_title(f"{CV_FOLDS}-Fold CV Score Distribution")
ax5.set_ylabel("Score")
ax5.set_ylim(0, 1.05)
 
plt.savefig("fraud_detection_results.png", dpi=150, bbox_inches="tight")
print("    Saved → fraud_detection_results.png")
plt.close()


    Saved → fraud_detection_results.png


C:\Users\Sahil Singh\AppData\Local\Temp\ipykernel_13484\37389551.py:6: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax5.boxplot(scores_df.values, labels=scores_df.columns, patch_artist=True,


In [16]:
#Model Serialisation
print("\n[9] Serialising model & scaler ...")
joblib.dump(clf, MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
print(f"    Model  saved → {MODEL_PATH}")
print(f"    Scaler saved → {SCALER_PATH}")


[9] Serialising model & scaler ...
    Model  saved → fraud_rf_model.joblib
    Scaler saved → fraud_scaler.joblib


In [17]:
#Inference Helper
print("\n[10] Demo: predicting on 5 random test samples ...")
 
sample_X     = X_test[:5]
sample_true  = y_test[:5]
sample_probs = clf.predict_proba(sample_X)[:, 1]
sample_preds = clf.predict(sample_X)
 
demo_df = pd.DataFrame({
    "True Label" : ["Fraud" if l else "Legit" for l in sample_true],
    "Predicted"  : ["Fraud" if l else "Legit" for l in sample_preds],
    "Fraud Prob" : [f"{p:.4f}" for p in sample_probs],
})
print(demo_df.to_string(index=False))
 
print("\n" + "=" * 60)
print("  Pipeline complete. Artifacts:")
print(f"   • {MODEL_PATH}")
print(f"   • {SCALER_PATH}")
print(f"   • fraud_detection_results.png")
print("=" * 60)


[10] Demo: predicting on 5 random test samples ...
True Label Predicted Fraud Prob
     Legit     Legit     0.0000
     Legit     Legit     0.0000
     Legit     Legit     0.0000
     Legit     Legit     0.0000
     Legit     Legit     0.0000

  Pipeline complete. Artifacts:
   • fraud_rf_model.joblib
   • fraud_scaler.joblib
   • fraud_detection_results.png
